# 02: exploring the Silver layer

Wildfire monitoring, Team 3, Workshop 2. Repository: https://github.com/adriangeb28/wildfire-monitoring

This notebook reads the Parquet files that the `silver_processing` DAG writes to `datalake_silver/` and checks
that they are what the preprocessing design promises: typed columns, UTC timestamps, Colombian regions only,
flags instead of deleted outliers, and every row traceable to its Bronze file.

Run it from the repository root after at least one Silver run (`docker compose up`, or
`python ingest.py --mode incremental --from-fixtures` on the host with `pip install -r requirements.txt`).

In [ ]:
from pathlib import Path
import json

import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SILVER = ROOT / "datalake_silver"
fire_files = sorted(SILVER.glob("fire_detections_*.parquet"))
weather_files = sorted(SILVER.glob("weather_hourly_*.parquet"))
rejected_files = sorted((SILVER / "_quarantine").glob("rejected_*.parquet"))
print("fire_detections files:", [p.name for p in fire_files])
print("weather_hourly files :", [p.name for p in weather_files])
print("quarantine files     :", [p.name for p in rejected_files])

## 1. One Silver file: schema, types and lineage metadata

In [ ]:
latest = fire_files[-1]
schema = pq.read_schema(latest)
print(schema)
print()
print(json.dumps(json.loads(schema.metadata[b"wildfire_pipeline"]), indent=2))

In [ ]:
fires = pd.read_parquet(latest)
print(fires.shape)
print(fires.dtypes.to_string())
fires.head()

## 2. All Silver runs together (what the Gold layer will do with PySpark in Workshop 3)

In [ ]:
all_fires = pd.concat([pd.read_parquet(p).assign(silver_file=p.name) for p in fire_files], ignore_index=True)
print("rows:", len(all_fires), "| unique event_id:", all_fires.event_id.nunique())
print(all_fires.groupby("silver_file").agg(rows=("event_id", "size"),
                                            first=("observed_at", "min"), last=("observed_at", "max"),
                                            bronze_files=("bronze_file", "nunique")).to_string())
print()
print("record_status:", all_fires.record_status.value_counts().to_dict())

## 3. Checks the preprocessing promises

In [ ]:
checks = {
    "observed_at is UTC": str(all_fires.observed_at.dt.tz) == "UTC",
    "every row has a Colombian region": all_fires.region_id.str.startswith("CO-").all(),
    "frp > 0 everywhere": bool((all_fires.frp > 0).all()),
    "no duplicated event_id inside a file": not all_fires.duplicated(["silver_file", "event_id"]).any(),
    "every row has its bronze_file": all_fires.bronze_file.notna().all(),
    "confidence is an ordered category": isinstance(all_fires.confidence_level.dtype, pd.CategoricalDtype),
}
for name, ok in checks.items():
    print(f"{'OK ' if ok else 'FAIL'} {name}")

## 4. A first look at the content

In [ ]:
print("Detections per region (all runs):")
print(all_fires.groupby("region_name").agg(detections=("event_id", "size"), frp_sum=("frp", "sum"),
                                            frp_max=("frp", "max"), outliers=("frp_outlier", "sum"))
      .sort_values("detections", ascending=False).head(10).round(1).to_string())
print()
print("Confidence levels:", all_fires.confidence_level.value_counts().to_dict())
print("FRP outliers flagged (kept):", int(all_fires.frp_outlier.sum()))

In [ ]:
ax = all_fires.set_index("observed_at").resample("6h").size().plot(figsize=(10, 3), marker="o")
ax.set_title("Detections per 6 hours in the Silver layer (Colombia)")
ax.set_ylabel("detections")

## 5. Weather and quarantine

In [ ]:
weather = pd.concat([pd.read_parquet(p) for p in weather_files], ignore_index=True)
print(weather.groupby("variable", observed=True).agg(rows=("value", "size"), nulls=("value", lambda s: int(s.isna().sum())),
                                                     imputed=("imputed", "sum"), mean=("value", "mean")).round(2).to_string())
print()
rejected = pd.concat([pd.read_parquet(p) for p in rejected_files], ignore_index=True) if rejected_files else pd.DataFrame()
if len(rejected):
    print("Quarantine by reason:", rejected.reason.value_counts().to_dict())
    print(json.loads(rejected.payload.iloc[0]))